### Ejemplo libro Geron para teoría

In [1]:
# Set working directory
import os
import pandas as pd

os.chdir("C:/Users/Usuario/Downloads")

#Parse dates identifica que se trata de fechas, lo cual vemos se hizo correctamente con df.info()
df = pd.read_csv("CTA_-_Ridership_-_Daily_Boarding_Totals.csv",parse_dates=["service_date"])
#Los datos son diarios
df

,service_date,day_type,bus,rail_boardings,total_rides
0,2001-01-01,U,297192,126455,423647
1,2001-01-02,W,780827,501952,1282779
2,2001-01-03,W,824923,536432,1361355
3,2001-01-04,W,870021,550011,1420032
4,2001-01-05,W,890426,557917,1448343
...,...,...,...,...,...
7696,2021-11-26,W,257700,189694,447394
7697,2021-11-27,A,237839,187065,424904
7698,2021-11-28,U,184817,147830,332647
7699,2021-11-29,W,421322,276090,697412


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7701 entries, 0 to 7700
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   service_date    7701 non-null   datetime64[ns]
 1   day_type        7701 non-null   object        
 2   bus             7701 non-null   int64         
 3   rail_boardings  7701 non-null   int64         
 4   total_rides     7701 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 300.9+ KB


In [2]:
#Cambiar el nombre de las columnas
df.columns = ["date", "day_type", "bus", "rail", "total"]  # shorter names
#Ordenar por fecha
df = df.sort_values("date").set_index("date")
#Quitar total que es la suma de dos columnas
df = df.drop("total", axis=1)  # no need for total, it's just bus + rail
df = df.drop_duplicates()  # remove duplicated months (2011-10 and 2014-07)

In [3]:
df.head()

,day_type,bus,rail
date,,,
2001-01-01,U,297192,126455
2001-01-02,W,780827,501952
2001-01-03,W,824923,536432
2001-01-04,W,870021,550011
2001-01-05,W,890426,557917


In [4]:
#Vamos a estudiar solo la columna rail. Como preprocesamiento, dividimos entre 1e6. Tomamos como entrenamiento
#el 2016 al 2018, como validación desde enero hasta finales de mayo (medio año) y prueba el resto

rail_train = df["rail"]["2016-01":"2018-12"] / 1e6
rail_valid = df["rail"]["2019-01":"2019-05"] / 1e6
rail_test = df["rail"]["2019-06":] / 1e6

In [5]:
import tensorflow as tf

#A continuaci\'on tomamos ventanas de tama\~no 56 cuyos valores son los inputs para una vez desdoblada la red
#tomar el valor de las observaciones una unidad en el tiempo futuro como outputs, siendo nuestro output real el valor de 
#la observaci\'on en el tiempo 57. De esta forma, al desdoblar la red las observaciones en el output 
#tienen el mismo tama\~o que la ventana para que haya contra quien asociar cada valor del input.
#Luego, tomamos un tama\~o de batch de 32 (se divide la muestra de miniseries en 32) aleatoriamente con una semilla
#de 42 para reproducibilidad.

#Tomamos un tamaño de batch de 32 y se revuelve con una semilla de 42. En cada batch vamos a tener muestras
#de las observaciones en el conjunto de entrenamiento, lo cual se refleja en que se tienen muestras de las ventanas. 

seq_length = 56
tf.random.set_seed(42)  # extra code – ensures reproducibility
train_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_train.to_numpy(),
    targets=rail_train[seq_length:],
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)

#Se utilizar\'a el modelo aprendido sobre los datos de validaci\'on, prediciendo a partir de 56 observaciones,
#el tama\~no de las ventana, la siguiente observaci\'on. Se incuye el tamaño del batch, 32, porque en la estimaci\'on
#final de cada observaci\'on hay que promediar las predicciones que se obtengan con los 32 distintos modelos aprendidos
valid_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_valid.to_numpy(),
    targets=rail_valid[seq_length:],
    sequence_length=seq_length,
    batch_size=32
)

In [7]:
#Agregar una neurona recurrente simple, o sea es una arquitectura en la cual tenemos un delay que corresponde a una
#capa oculta, los inputs están dadas por los valores de las ventanas y los outputs son los valores sobre los que se predice

tf.random.set_seed(42)  # extra code – ensures reproducibility
model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(1, input_shape=[None, 1])
])

C:\Users\Usuario\anaconda3\lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [8]:
# extra code – defines a utility function we'll reuse several time

def fit_and_evaluate(model, train_set, valid_set, learning_rate, epochs=500):
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=50, restore_best_weights=True)
    opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
    history = model.fit(train_set, validation_data=valid_set, epochs=epochs,
                        callbacks=[early_stopping_cb])
    valid_loss, valid_mae = model.evaluate(valid_set)
    return valid_mae * 1e6

In [9]:
##Ajuste 

fit_and_evaluate(model, train_ds, valid_ds, learning_rate=0.02)

Epoch 1/500
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0618 - mae: 0.3113 - val_loss: 0.0120 - val_mae: 0.1162
Epoch 2/500
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0122 - mae: 0.1236 - val_loss: 0.0120 - val_mae: 0.1157
Epoch 3/500
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0116 - mae: 0.1236 - val_loss: 0.0115 - val_mae: 0.1170
Epoch 4/500
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0116 - mae: 0.1262 - val_loss: 0.0115 - val_mae: 0.1164
Epoch 5/500
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0117 - mae: 0.1272 - val_loss: 0.0115 - val_mae: 0.1164
Epoch 6/500
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0114 - mae: 0.1247 - val_loss: 0.0114 - val_mae: 0.1164
Epoch 7/500
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0114 - mae: 0.1255 - val_loss: 0.0114 - val_mae: 0.1159
Epoch 8/500
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0110 - mae: 0.1224 - val_loss: 0.0113 - val_mae: 0.1158
Epoch 9/500
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.01

102779.90996837616